## RAG Pipeline -  Data Ingestion to Vector DB Pipeline

In [1]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

e:\Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Finding all the pdfs recursively 
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Adding source information in the metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f" Error {e}")
    print(f"\nTotal Documets loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: Data Structures and Algorithms Explained_.pdf
Loaded 36 pages

Processing: Ed 5, Gilbert Strang - Introduction to Linear Algebra (2016, Wellesley-Cambridge Press).pdf
Loaded 584 pages

Processing: Linear Regression_ Comprehensive Guide_.pdf
Loaded 44 pages

Processing: ML Machine Learning-A Probabilistic Perspective.pdf
Loaded 1098 pages

Processing: Shivanshu-Mishra-Resume .pdf
Loaded 2 pages

Total Documets loaded: 1764


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m136 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Data Structures and Algorithms Explained', 'source': '..\\data\\Pdf\\Data Structures and Algorithms Explained_.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Data Structures and Algorithms Explained_.pdf', 'file_type': 'pdf'}, page_content="Understanding  Fundamental  Data  Structures  and  Algorithms  1.  Introduction  to  Data  Structures  and  Algorithms  In  the  realm  of  computer  science,  the  efficient  organization  and  manipulation  of  data  \nare\n \nparamount.\n \nThis\n \nis\n \nachieved\n \nthrough\n \nthe\n \nuse\n \nof\n \ndata\n \nstructures\n \nand\n \nalgorithms.\n \nData\n \nstructures\n \nserve\n \nas\n \nthe\n \nfoundational\n \nbuilding\n \nblocks\n \nfor\n \nstoring\n \nand\n \norganizing\n \ninformation\n \nin\n \na\n \ncomputer's\n \nmemory,\n \nenabling\n \nefficient\n \naccess\n \nand\n \nmodification\n \n2\n.  \

## TEXT SPLITTING GET INTO CHUNKS 

In [4]:
def split_documents(documents, chunk_size = 1000, chunk_overlap=200):
    """ Split documents into smaller chunks for better RAG performance """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n," "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Example
    if split_docs:
        print(f"\nExample Chunk :" )
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

Split 1764 documents into 5080 chunks

Example Chunk :
Content: Understanding  Fundamental  Data  Structures  and  Algorithms  1.  Introduction  to  Data  Structures  and  Algorithms  In  the  realm  of  computer  science,  the  efficient  organization  and  manip...
Metadata: {'producer': 'Skia/PDF m136 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Data Structures and Algorithms Explained', 'source': '..\\data\\Pdf\\Data Structures and Algorithms Explained_.pdf', 'total_pages': 36, 'page': 0, 'page_label': '1', 'source_file': 'Data Structures and Algorithms Explained_.pdf', 'file_type': 'pdf'}


## Embedding and VectorStoreDB

In [6]:
import numpy as numpy
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity